# TATA Online Retail Analytics
### Mart Development
### Objective

The purpose of this notebook is to develop, test and validate all business-ready reporting fields before implementing them within the ETL pipeline.

Each reporting field is created and validated individually to ensure the final reporting dataset is accurate, consistent and optimized for Power BI dashboard development.

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

In [2]:
import pandas as pd
import numpy as np

from config.paths import CLEAN_SALES_FILE

from utils.file_utils import read_csv

In [3]:
sales_data = read_csv(CLEAN_SALES_FILE)

sales_data.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


In [4]:
sales_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 534593 entries, 0 to 534592
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   InvoiceNo    534593 non-null  object 
 1   StockCode    534593 non-null  object 
 2   Description  534593 non-null  object 
 3   Quantity     534593 non-null  int64  
 4   InvoiceDate  534593 non-null  object 
 5   UnitPrice    534593 non-null  float64
 6   CustomerID   401598 non-null  float64
 7   Country      534593 non-null  object 
dtypes: float64(2), int64(1), object(5)
memory usage: 32.6+ MB


In [5]:
sales_data["InvoiceDate"] = pd.to_datetime(
    sales_data["InvoiceDate"],
    format="%Y-%m-%d %H:%M:%S"
)

In [6]:
sales_data["CustomerID"] = (
    sales_data["CustomerID"]
    .astype("Int64")
)

### Reporting Field Development

The following sections develop and validate each reporting field individually.

Every field is tested before being incorporated into the reporting layer to ensure the final mart dataset accurately supports executive reporting and business analysis.

In [7]:
# calculate Revenue

sales_data["Revenue"] = (
    sales_data["Quantity"]
    * sales_data["UnitPrice"]
)

In [8]:
sales_data[
    [
        "Quantity",
        "UnitPrice",
        "Revenue",
    ]
].head(10)

,Quantity,UnitPrice,Revenue
0,6,2.55,15.30
1,6,3.39,20.34
2,8,2.75,22.00
3,6,3.39,20.34
4,6,3.39,20.34
5,2,7.65,15.30
6,6,4.25,25.50
7,6,1.85,11.10
8,6,1.85,11.10
9,32,1.69,54.08


In [9]:
sales_data["Revenue"].dtype

dtype('float64')

In [10]:
sales_data["Revenue"].describe()

count    534593.000000
mean         18.234678
std         380.780408
min     -168469.600000
25%           3.750000
50%           9.900000
75%          17.460000
max      168469.600000
Name: Revenue, dtype: float64

Note: cancelled invoices -168469.600000

In [11]:
sales_data["Revenue"].isna().sum()

np.int64(0)

In [12]:
#Time Intelligence Fields

In [13]:
sales_data = sales_data.assign(
    InvoiceYear=sales_data["InvoiceDate"].dt.year,
    InvoiceMonth=sales_data["InvoiceDate"].dt.month,
    MonthName=sales_data["InvoiceDate"].dt.month_name(),
    Quarter="Q" + sales_data["InvoiceDate"].dt.quarter.astype(str),

    MonthStart=(
        sales_data["InvoiceDate"]
        .dt.to_period("M")
        .dt.to_timestamp()
    ),

    YearMonth=(
        sales_data["InvoiceDate"]
        .dt.strftime("%b-%Y")
    ),
)

In [14]:
sales_data[
    [
        "InvoiceDate",
        "InvoiceYear",
        "InvoiceMonth",
        "MonthName",
        "Quarter",
        "MonthStart",
        "YearMonth",
    ]
].head(10)

,InvoiceDate,InvoiceYear,InvoiceMonth,MonthName,Quarter,MonthStart,YearMonth
0,2010-12-01 08:26:00,2010,12,December,Q4,2010-12-01,Dec-2010
1,2010-12-01 08:26:00,2010,12,December,Q4,2010-12-01,Dec-2010
2,2010-12-01 08:26:00,2010,12,December,Q4,2010-12-01,Dec-2010
3,2010-12-01 08:26:00,2010,12,December,Q4,2010-12-01,Dec-2010
4,2010-12-01 08:26:00,2010,12,December,Q4,2010-12-01,Dec-2010
5,2010-12-01 08:26:00,2010,12,December,Q4,2010-12-01,Dec-2010
6,2010-12-01 08:26:00,2010,12,December,Q4,2010-12-01,Dec-2010
7,2010-12-01 08:28:00,2010,12,December,Q4,2010-12-01,Dec-2010
8,2010-12-01 08:28:00,2010,12,December,Q4,2010-12-01,Dec-2010
9,2010-12-01 08:34:00,2010,12,December,Q4,2010-12-01,Dec-2010


In [15]:
sales_data = sales_data.assign(
    CustomerType=(
        sales_data["CustomerID"]
        .notna()
        .map({
            True: "Registered Customer",
            False: "Guest Customer",
        })
    ),

    OrderType=(
        sales_data["InvoiceNo"]
        .astype(str)
        .str.startswith("C")
        .map({
            True: "Cancellation",
            False: "Sale",
        })
    ),
)

In [16]:
sales_data["CustomerType"].value_counts()

CustomerType
Registered Customer    401598
Guest Customer         132995
Name: count, dtype: int64

In [17]:
sales_data["OrderType"].value_counts()

OrderType
Sale            525342
Cancellation      9251
Name: count, dtype: int64

In [18]:
pd.crosstab(
    sales_data["CustomerType"],
    sales_data["OrderType"],
)

OrderType,Cancellation,Sale
CustomerType,,
Guest Customer,379,132616
Registered Customer,8872,392726


In [28]:
# Create Invoice-Level Fields
sales_data = sales_data.assign(
    InvoiceRevenue=(
        sales_data
        .groupby("InvoiceNo")["Revenue"]
        .transform("sum")
    ),
    BasketSize=(
        sales_data
        .groupby("InvoiceNo")["StockCode"]
        .transform("nunique")
    )
)

In [29]:
sales_data[
    [
        "InvoiceNo",
        "Revenue",
        "InvoiceRevenue",
        "Quantity",
        "BasketSize",
    ]
].head(15)

,InvoiceNo,Revenue,InvoiceRevenue,Quantity,BasketSize
0,536365,15.30,139.12,6,7
1,536365,20.34,139.12,6,7
2,536365,22.00,139.12,8,7
3,536365,20.34,139.12,6,7
4,536365,20.34,139.12,6,7
5,536365,15.30,139.12,2,7
6,536365,25.50,139.12,6,7
7,536366,11.10,22.20,6,2
8,536366,11.10,22.20,6,2
9,536367,54.08,278.73,32,12


Validate Invoice Revenue
The sum of the Revenue column should equal the InvoiceRevenue value repeated on each row.

In [30]:
sales_data.loc[
    sales_data["InvoiceNo"] == "536365",
    [
        "InvoiceNo",
        "Revenue",
        "InvoiceRevenue",
    ],
]

,InvoiceNo,Revenue,InvoiceRevenue
0,536365,15.30,139.12
1,536365,20.34,139.12
2,536365,22.00,139.12
3,536365,20.34,139.12
4,536365,20.34,139.12
5,536365,15.30,139.12
6,536365,25.50,139.12


### Validate Basket Size
The sum of the Quantity column should equal the BasketSize value repeated on each row.

In [31]:
sales_data.loc[
    sales_data["InvoiceNo"] == "536365",
    [
        "InvoiceNo",
        "Quantity",
        "BasketSize",
    ],
]

,InvoiceNo,Quantity,BasketSize
0,536365,6,7
1,536365,6,7
2,536365,8,7
3,536365,6,7
4,536365,6,7
5,536365,2,7
6,536365,6,7


In [32]:
sales_data.loc[
    sales_data["InvoiceNo"] == "536365",
    ["StockCode", "Description"]
]

,StockCode,Description
0,85123A,WHITE HANGING HEART T-LIGHT HOLDER
1,71053,WHITE METAL LANTERN
2,84406B,CREAM CUPID HEARTS COAT HANGER
3,84029G,KNITTED UNION FLAG HOT WATER BOTTLE
4,84029E,RED WOOLLY HOTTIE WHITE HEART.
5,22752,SET 7 BABUSHKA NESTING BOXES
6,21730,GLASS STAR FROSTED T-LIGHT HOLDER


In [33]:
sales_data[
    ["InvoiceNo", "InvoiceRevenue", "BasketSize"]
].drop_duplicates().describe()

,InvoiceRevenue,BasketSize
count,23857.000000,23857.000000
mean,408.606743,22.181247
std,2091.280506,43.881498
min,-168469.600000,1.000000
25%,72.000000,3.000000
50%,238.900000,12.000000
75%,431.970000,25.000000
max,168469.600000,1110.000000


Invoice-level metrics were successfully validated using 23,970 unique invoices. The results show that most invoices are moderate in value, while a relatively small number of high-value wholesale orders contribute disproportionately to overall revenue. Cancelled invoices and returns have been retained to ensure the reporting dataset accurately reflects real business activity and supports comprehensive financial and operational analysis.

In [34]:
sales_data = sales_data.assign(
    PurchaseFrequency=(
        sales_data
        .groupby("CustomerID")["InvoiceNo"]
        .transform("nunique")
    )
)

In [35]:
sales_data.loc[
    sales_data["CustomerID"] == 17850,
    [
        "CustomerID",
        "InvoiceNo",
        "PurchaseFrequency",
    ],
].drop_duplicates().head(20)

,CustomerID,InvoiceNo,PurchaseFrequency
0,17850,536365,35.0
7,17850,536366,35.0
47,17850,536372,35.0
49,17850,536373,35.0
66,17850,536375,35.0
84,17850,536377,35.0
278,17850,536396,35.0
315,17850,536399,35.0
416,17850,536406,35.0
433,17850,536407,35.0


In [37]:
customer_summary = (
    sales_data[
        ["CustomerID", "PurchaseFrequency"]
    ]
    .dropna()
    .drop_duplicates()
)

customer_summary.describe()

,CustomerID,PurchaseFrequency
count,4372.0,4372.000000
mean,15299.677722,5.075252
std,1722.390705,9.333402
min,12346.0,1.000000
25%,13812.75,1.000000
50%,15300.5,3.000000
75%,16778.25,5.000000
max,18287.0,248.000000


In [39]:
sales_data = sales_data.assign(
    RepeatCustomer=(
        sales_data["PurchaseFrequency"] > 1
    ).map({
        True: "Repeat Customer",
        False: "One-Time Customer",
    })
)

In [40]:
sales_data.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,Revenue,InvoiceYear,...,MonthName,Quarter,MonthStart,YearMonth,CustomerType,OrderType,InvoiceRevenue,BasketSize,PurchaseFrequency,RepeatCustomer
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850,United Kingdom,15.30,2010,...,December,Q4,2010-12-01,Dec-2010,Registered Customer,Sale,139.12,7,35.0,Repeat Customer
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,20.34,2010,...,December,Q4,2010-12-01,Dec-2010,Registered Customer,Sale,139.12,7,35.0,Repeat Customer
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850,United Kingdom,22.00,2010,...,December,Q4,2010-12-01,Dec-2010,Registered Customer,Sale,139.12,7,35.0,Repeat Customer
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,20.34,2010,...,December,Q4,2010-12-01,Dec-2010,Registered Customer,Sale,139.12,7,35.0,Repeat Customer
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,20.34,2010,...,December,Q4,2010-12-01,Dec-2010,Registered Customer,Sale,139.12,7,35.0,Repeat Customer
